# Gráficos do pôster — Ensino Médio, escolas integrais e rendimento (2011–2025)

Este notebook lê a base consolidada (`bd5.csv`, colunas do dicionário `dicionario_dados_educaionais_2011a2025.xlsx`) e gera:

1. **Expansão das escolas integrais por região** — versão em % e em número absoluto (linhas), mais três formas alternativas: mapa de calor, barras empilhadas e "halteres" (2011 → 2025).
2. **Taxas de reprovação e abandono por modalidade, ano a ano** — versão em % (taxa agregada) e em número absoluto de alunos.
3. **Boxplots de reprovação e abandono** por dependência administrativa e modalidade, comparando **pré-pandemia (2016–2019)** e **pós-pandemia (2022–2025)**.

As figuras saem em PNG (300 dpi) e SVG na pasta definida em `PASTA_SAIDA`, junto com as tabelas (CSV) que dão origem a cada gráfico.

**Decisões que valem para todo o notebook**

| Item | O que foi feito |
|---|---|
| Etapa | Só Ensino Médio (colunas `*_EM_*`) e só escolas-ano com matrícula no EM (`QT_MAT_EM_REG + QT_MAT_EM_INT > 0`). |
| Rendimento | Mantidas só as escolas-ano com `QT_APR_EM_TOTAL`, `QT_REP_EM_TOTAL` e `QT_ABN_EM_TOTAL` preenchidos. Zero é válido; só `NaN` é descartado. |
| Modalidade | `ENS_PRED_EM` da própria base (Regular < 48%, Misto 48–52%, Integral > 52% de matrículas integrais). |
| Dependência | Estadual e Municipal agrupadas em "Estadual/Municipal", como no relatório da base. |
| Taxa de uma escola | reprovados ÷ (aprovados + reprovados + abandonos) × 100 — a mesma fórmula do INEP. |
| Pandemia | 2020–2021 sombreados nos gráficos de linha. |

> Atenção: na base, as quantidades de aprovados, reprovados e abandonos são **estimadas** (taxa do INEP × matrículas do Censo, arredondadas). Os gráficos "absolutos" mostram essas estimativas.

## 0. Configuração

Tudo o que pode mudar está nesta célula. O resto do notebook não precisa ser editado.

In [ ]:
# ===================== CONFIGURAÇÃO =====================
CAMINHO_BASE = "bd5.csv"         # caminho da base (na mesma pasta do notebook ou caminho completo)
PASTA_SAIDA  = "figuras_poster"  # figuras (PNG 300 dpi + SVG) e tabelas (CSV)
IDIOMA       = "en"              # "en" para o pôster; "pt" para conferir em português

# Gráfico 1 — dependências contadas (o gráfico do PDF usava só Estadual/Municipal)
DEPS_GRAFICO1   = ["Estadual/Municipal"]
MOSTRAR_BRASIL  = True                 # linha/linha de referência com o total do Brasil
ANOS_HALTERES   = (2011, 2025)         # anos comparados no gráfico 1e

# Gráfico 2 — dependências incluídas nas taxas agregadas (o PDF usava todas)
DEPS_GRAFICO2 = ["Federal", "Estadual/Municipal", "Privada"]

# Gráfico 3 — boxplots
PERIODOS = {"pre": (2016, 2019), "pos": (2022, 2025)}   # 4 anos cada
UNIDADE_BOX   = "escola-ano"   # "escola-ano" ou "escola-periodo" (taxa acumulada da escola nos 4 anos)
PESO_BOXPLOT  = "alunos"       # "alunos" = escola pesa pelo nº de alunos; None = cada unidade pesa 1
MIN_MATRICULAS_BOXPLOT = 0     # ex.: 51 para tirar a faixa "até 50 matrículas"
MIN_N_CELULA  = 30             # caixas com menos unidades que isso ficam hachuradas

ANOS_PANDEMIA = (2020, 2021)

## 1. Bibliotecas, cores e textos

In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

SAIDA = Path(PASTA_SAIDA)
SAIDA.mkdir(parents=True, exist_ok=True)

DEPS_TODAS  = ["Federal", "Estadual/Municipal", "Privada"]
REGIOES     = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
MODALIDADES = ["Regular", "Misto", "Integral"]

# Paleta categórica em ordem fixa (pares vizinhos separáveis para daltonismo)
COR_REGIAO  = dict(zip(REGIOES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
COR_MOD     = {"Regular": "#2a78d6", "Misto": "#eb6834", "Integral": "#1baf7a"}
COR_PERIODO = {"pre": "#9ec5f4", "pos": "#1c5cab"}          # mesmo azul, claro → escuro
RAMPA_AZUL  = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
               "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
TINTA, TINTA_2, APAGADO = "#0b0b0b", "#52514e", "#898781"
GRADE, EIXO, SOMBRA = "#e1e0d9", "#c3c2b7", "#efeee9"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "figure.facecolor": "white",
    "font.family": "sans-serif", "font.size": 11,
    "axes.edgecolor": EIXO, "axes.linewidth": 0.8, "axes.labelcolor": TINTA_2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRADE, "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "xtick.color": EIXO, "ytick.color": EIXO, "xtick.labelcolor": TINTA_2, "ytick.labelcolor": TINTA_2,
    "legend.frameon": False, "lines.linewidth": 2.0, "lines.markersize": 4.5,
    "svg.fonttype": "none",
})

TXT = {
    "en": {
        "ano": "Year", "regiao": "Region", "brasil": "Brazil",
        "mod": {"Regular": "Regular", "Misto": "Mixed", "Integral": "Full-time"},
        "dep": {"Federal": "Federal", "Estadual/Municipal": "State/Municipal", "Privada": "Private"},
        "periodo": {"pre": "Pre-pandemic", "pos": "Post-pandemic"},
        "pandemia": "2020–2021",
        "taxa": {"TX_REP": "Failure", "TX_ABN": "Dropout"},
        "g1_titulo_pct": "Share of {escolas} classified as full-time",
        "g1_titulo_abs": "Number of {escolas} classified as full-time",
        "g1_y_pct": "Full-time schools (% of the region's schools)",
        "g1_y_abs": "Full-time schools",
        "g1_nota_pct": "Share = full-time schools ÷ {escolas} with pass/fail/dropout data, in each region and year.",
        "g1_nota_abs": "Count of {escolas} classified as full-time (more than 52% of high school enrollment in full-time classes).",
        "g1_heat_titulo": "Share of {escolas} classified as full-time (%)",
        "g1_halter_titulo": "Share of {escolas} classified as full-time, {a0} vs {a1}",
        "g1_halter_x": "Full-time schools (% of the region's schools)",
        "pp": "p.p.",
        "g2_titulo_pct": "Failure and dropout rates by school schedule",
        "g2_titulo_abs": "Students who failed or dropped out, by school schedule",
        "g2_sub": "{taxa}",
        "g2_y_pct": "Rate (%)",
        "g2_y_abs": "Students (thousands)",
        "g2_nota_pct": "Aggregate rate = students who failed (or dropped out) ÷ students with a final status (passed + failed + dropped out), summed over {escolas}. Shading: 2020–2021.",
        "g2_nota_abs": "Estimated counts (school rate × enrollment, INEP). Totals reflect both the rate and the size of each schedule group. Shading: 2020–2021.",
        "g3_titulo": "High school failure and dropout rates by school schedule and administrative category",
        "g3_y": "{taxa} rate (%)",
        "g3_nota_peso": "Each box: distribution of school rates weighted by the school's number of high school students. ",
        "g3_nota_sem_peso": "Each box: distribution of school rates, each {unidade} counted once. ",
        "g3_nota_fim": "Whiskers: most extreme value within 1.5 × IQR; outliers hidden. Hatched boxes: fewer than {n} {unidade}s.",
        "unidade": {"escola-ano": "school-year", "escola-periodo": "school-period"},
        "n_baixo": "n < {n}",
        "deps_txt": {"Federal": "federal", "Estadual/Municipal": "state/municipal", "Privada": "private"},
        "escolas_de": "{deps} high schools",
        "escolas_todas": "high schools (all administrative categories)",
    },
    "pt": {
        "ano": "Ano", "regiao": "Região", "brasil": "Brasil",
        "mod": {"Regular": "Regular", "Misto": "Misto", "Integral": "Integral"},
        "dep": {"Federal": "Federal", "Estadual/Municipal": "Estadual/Municipal", "Privada": "Privada"},
        "periodo": {"pre": "Pré-pandemia", "pos": "Pós-pandemia"},
        "pandemia": "2020–2021",
        "taxa": {"TX_REP": "Reprovação", "TX_ABN": "Abandono"},
        "g1_titulo_pct": "Proporção de {escolas} classificadas como integrais",
        "g1_titulo_abs": "Número de {escolas} classificadas como integrais",
        "g1_y_pct": "Escolas integrais (% das escolas da região)",
        "g1_y_abs": "Escolas integrais",
        "g1_nota_pct": "Proporção = escolas integrais ÷ {escolas} com rendimento informado, em cada região e ano.",
        "g1_nota_abs": "Contagem de {escolas} classificadas como integrais (mais de 52% das matrículas do EM em tempo integral).",
        "g1_heat_titulo": "Proporção de {escolas} classificadas como integrais (%)",
        "g1_halter_titulo": "Proporção de {escolas} classificadas como integrais, {a0} vs {a1}",
        "g1_halter_x": "Escolas integrais (% das escolas da região)",
        "pp": "p.p.",
        "g2_titulo_pct": "Taxas de reprovação e abandono por modalidade",
        "g2_titulo_abs": "Alunos reprovados e que abandonaram, por modalidade",
        "g2_sub": "{taxa}",
        "g2_y_pct": "Taxa (%)",
        "g2_y_abs": "Alunos (milhares)",
        "g2_nota_pct": "Taxa agregada = soma dos reprovados (ou que abandonaram) ÷ soma dos alunos com situação final (aprovados + reprovados + abandonos), {escolas}. Sombra: 2020–2021.",
        "g2_nota_abs": "Quantidades estimadas (taxa da escola × matrículas, INEP). O total reflete a taxa e o tamanho de cada modalidade. Sombra: 2020–2021.",
        "g3_titulo": "Reprovação e abandono no Ensino Médio por modalidade e dependência administrativa",
        "g3_y": "{taxa} (%)",
        "g3_nota_peso": "Cada caixa: distribuição das taxas das escolas, com peso pelo número de alunos do EM. ",
        "g3_nota_sem_peso": "Cada caixa: distribuição das taxas das escolas, cada {unidade} conta uma vez. ",
        "g3_nota_fim": "Bigodes: valor mais extremo dentro de 1,5 × IIQ; outliers ocultos. Caixas hachuradas: menos de {n} unidades ({unidade}).",
        "unidade": {"escola-ano": "escola-ano", "escola-periodo": "escola-período"},
        "n_baixo": "n < {n}",
        "deps_txt": {"Federal": "federais", "Estadual/Municipal": "estaduais/municipais", "Privada": "privadas"},
        "escolas_de": "escolas de EM {deps}",
        "escolas_todas": "escolas de EM (todas as dependências)",
    },
}
T = TXT[IDIOMA]


def nome_deps(deps):
    """Frase com as dependências escolhidas, usada em títulos e notas."""
    if set(deps) == set(DEPS_TODAS):
        return T["escolas_todas"]
    return T["escolas_de"].format(deps=" + ".join(T["deps_txt"][d] for d in deps))


PCT = mticker.FuncFormatter(lambda v, _: f"{v:g}%" if v >= 0 else "")
MIL = mticker.StrMethodFormatter("{x:,.0f}")


def sombra_pandemia(ax):
    ax.axvspan(ANOS_PANDEMIA[0] - 0.5, ANOS_PANDEMIA[1] + 0.5, color=SOMBRA, lw=0, zorder=0)


def eixo_anos(ax, anos):
    ax.set_xticks(range(min(anos), max(anos) + 1, 2))
    ax.set_xlim(min(anos) - 0.4, max(anos) + 0.4)
    ax.xaxis.set_minor_locator(mticker.NullLocator())


def rotulos_finais(ax, x, valores, textos, folga=0.055):
    """Rótulos à direita do último ponto, afastados entre si para não se sobreporem."""
    y0, y1 = ax.get_ylim()
    passo = folga * (y1 - y0)
    itens = sorted(zip(valores, textos))
    posicoes = []
    for v, _ in itens:
        posicoes.append(v if not posicoes or v - posicoes[-1] >= passo else posicoes[-1] + passo)
    for (v, txt), y in zip(itens, posicoes):
        ax.annotate(txt, xy=(x, v), xytext=(x + 0.45, y), textcoords="data", va="center", ha="left",
                    fontsize=9, color=TINTA_2, annotation_clip=False)


def nota(fig, texto, y=-0.01, largura=150):
    fig.text(0.01, y, textwrap.fill(texto, largura), ha="left", va="top", fontsize=8, color=APAGADO)


def salvar(fig, nome):
    for ext in ("png", "svg"):
        fig.savefig(SAIDA / f"{nome}.{ext}", bbox_inches="tight", facecolor="white")
    print(f"salvo: {SAIDA / nome}.png / .svg")

## 2. Leitura da base

Só as colunas usadas são carregadas (a base completa tem ~2 milhões de linhas). O separador é detectado pelo cabeçalho.

In [ ]:
COLS_NUM = ["QT_MAT_EM_REG", "QT_MAT_EM_INT", "QT_APR_EM_TOTAL", "QT_REP_EM_TOTAL", "QT_ABN_EM_TOTAL"]
COLS = ["ANO", "CO_ESCOLA", "REGIAO", "DEP_ADM", "ENS_PRED_EM"] + COLS_NUM

with open(CAMINHO_BASE, encoding="utf-8-sig") as f:
    cabecalho = f.readline()
SEP = ";" if cabecalho.count(";") > cabecalho.count(",") else ","

bruta = pd.read_csv(CAMINHO_BASE, sep=SEP, encoding="utf-8-sig", low_memory=False,
                    usecols=lambda c: c.strip() in COLS)
bruta.columns = bruta.columns.str.strip()
faltando = set(COLS) - set(bruta.columns)
assert not faltando, f"Colunas não encontradas na base: {faltando}"

# números: aceita vírgula decimal, se houver
for c in COLS_NUM + ["ANO"]:
    if not pd.api.types.is_numeric_dtype(bruta[c]):
        bruta[c] = pd.to_numeric(bruta[c].astype("string").str.replace(",", ".", regex=False), errors="coerce")
bruta["ANO"] = bruta["ANO"].astype(int)

# textos padronizados
bruta["REGIAO"] = bruta["REGIAO"].astype("string").str.strip().str.replace("_", "-", regex=False).str.title()
bruta["ENS_PRED_EM"] = bruta["ENS_PRED_EM"].astype("string").str.strip().str.title()
MAPA_DEP = {"federal": "Federal", "1": "Federal",
            "estadual": "Estadual/Municipal", "2": "Estadual/Municipal",
            "municipal": "Estadual/Municipal", "3": "Estadual/Municipal",
            "privada": "Privada", "4": "Privada"}
bruta["DEP"] = bruta["DEP_ADM"].astype("string").str.strip().str.lower().map(MAPA_DEP)

print(f"Separador: '{SEP}'  |  linhas: {len(bruta):,}  |  anos: {bruta['ANO'].min()}–{bruta['ANO'].max()}")
print("Regiões:", sorted(bruta["REGIAO"].dropna().unique()))
print("Dependências (original → agrupada):")
print(bruta.groupby("DEP_ADM", dropna=False)["DEP"].first())

## 3. Filtro: Ensino Médio com rendimento informado

A tabela abaixo registra quantas escolas-ano saem em cada passo (útil para o texto de metodologia).
Escolas-ano com aprovados + reprovados + abandonos = 0 **ficam** na base (zero é válido): contam como escola nos gráficos de contagem, mas não têm taxa própria (0 ÷ 0), então não entram nos boxplots.

In [ ]:
REND = ["QT_APR_EM_TOTAL", "QT_REP_EM_TOTAL", "QT_ABN_EM_TOTAL"]
passos = [("Linhas na base (escolas-ano, todas as etapas)", len(bruta))]

bruta["MAT_EM"] = bruta[["QT_MAT_EM_REG", "QT_MAT_EM_INT"]].sum(axis=1, min_count=1)
em = bruta[bruta["MAT_EM"] > 0]
passos.append(("Com matrícula no Ensino Médio", len(em)))

sem_rend = em[REND].isna().any(axis=1)
passos.append(("   descartadas: aprovação, reprovação ou abandono = NaN", int(sem_rend.sum())))
em = em[~sem_rend]

sem_mod = ~em["ENS_PRED_EM"].isin(MODALIDADES).fillna(False)
sem_dep = em["DEP"].isna()
sem_reg = ~em["REGIAO"].isin(REGIOES).fillna(False)
passos.append(("   descartadas: modalidade (ENS_PRED_EM) ausente", int(sem_mod.sum())))
passos.append(("   descartadas: dependência ou região não reconhecida", int((~sem_mod & (sem_dep | sem_reg)).sum())))
df = em[~sem_mod & ~sem_dep & ~sem_reg].copy()
passos.append(("Escolas-ano analisadas", len(df)))

df["ALUNOS"] = df[REND].sum(axis=1)               # alunos com situação final
denominador = df["ALUNOS"].where(df["ALUNOS"] > 0)
df["TX_REP"] = 100 * df["QT_REP_EM_TOTAL"] / denominador
df["TX_ABN"] = 100 * df["QT_ABN_EM_TOTAL"] / denominador
passos.append(("   dessas, com aprovados + reprovados + abandonos = 0 (sem taxa própria)", int((df["ALUNOS"] == 0).sum())))

duplicadas = int(df.duplicated(["ANO", "CO_ESCOLA"]).sum())
passos.append(("   linhas repetidas (mesma escola no mesmo ano)", duplicadas))

resumo_filtro = pd.DataFrame(passos, columns=["passo", "escolas-ano"]).set_index("passo")
resumo_filtro.to_csv(SAIDA / "tab0_filtro.csv", encoding="utf-8-sig")
display(resumo_filtro)
print(f"Escolas distintas: {df['CO_ESCOLA'].nunique():,}")

display(pd.crosstab(df["ANO"], df["ENS_PRED_EM"]).reindex(columns=MODALIDADES, fill_value=0)
        .assign(Total=lambda t: t.sum(axis=1)))

## 4. Porte das escolas e a escolha do peso

**O problema.** Numa escola com 8 alunos no EM, um único reprovado vira 12,5% de reprovação. Escolas pequenas produzem taxas extremas por acaso e alargam as caixas dos boxplots.

**Classificação oficial encontrada.** O INEP não publica uma classificação de porte isolada; o porte aparece como um dos quatro componentes do **Indicador de Complexidade de Gestão da Escola** (Nota Técnica INEP, 2014), medido pelo número de matrículas de escolarização, em seis faixas: **até 50; 51–150; 151–300; 301–500; 501–1.000; mais de 1.000**. Usamos essas faixas aplicadas às **matrículas do Ensino Médio** — isso é uma adaptação nossa: na nota técnica o porte se refere ao total de matrículas da escola, e a base não tem todas as etapas (infantil, EJA, profissional).

**O que fazer com o peso, gráfico a gráfico**

- **Gráfico 2 (linhas de taxa):** não precisa de peso extra. A taxa agregada (soma dos reprovados ÷ soma dos alunos) já é uma média ponderada pelo número de alunos: uma escola de 10 alunos pesa 10, uma de 1.000 pesa 1.000.
- **Gráfico 3 (boxplots):** aqui cada escola contaria igual. Sugestão: **ponderar cada escola pelo número de alunos** (`PESO_BOXPLOT = "alunos"`). A leitura da caixa passa a ser "taxa da escola em que o aluno estuda". Alternativa mais simples de explicar: sem peso, mas tirando a faixa "até 50" (`MIN_MATRICULAS_BOXPLOT = 51`). A tabela de sensibilidade no fim do notebook mostra se a escolha muda a conclusão.
- **Gráfico 1 (contagem de escolas):** a unidade é a escola, então não se aplica peso.

A tabela abaixo mostra a dispersão das taxas por faixa de porte — se as faixas pequenas tiverem caixas (P25–P75) e P90 bem maiores, o argumento do peso está confirmado nos dados de vocês.

In [ ]:
FAIXAS = [0, 50, 150, 300, 500, 1000, np.inf]
ROTULOS_PORTE = ["até 50", "51–150", "151–300", "301–500", "501–1.000", "mais de 1.000"]
df["PORTE"] = pd.cut(df["MAT_EM"], FAIXAS, labels=ROTULOS_PORTE)

com_taxa = df[df["ALUNOS"] > 0]
tab_porte = com_taxa.groupby("PORTE", observed=False).agg(
    escolas_ano=("CO_ESCOLA", "size"),
    alunos=("ALUNOS", "sum"),
    rep_P25=("TX_REP", lambda s: s.quantile(.25)),
    rep_mediana=("TX_REP", "median"),
    rep_P75=("TX_REP", lambda s: s.quantile(.75)),
    rep_P90=("TX_REP", lambda s: s.quantile(.90)),
    abn_mediana=("TX_ABN", "median"),
    abn_P90=("TX_ABN", lambda s: s.quantile(.90)),
    pct_rep_zero=("TX_REP", lambda s: 100 * (s == 0).mean()),
)
tab_porte.insert(1, "% escolas-ano", 100 * tab_porte["escolas_ano"] / tab_porte["escolas_ano"].sum())
tab_porte.insert(3, "% alunos", 100 * tab_porte["alunos"] / tab_porte["alunos"].sum())
tab_porte.to_csv(SAIDA / "tab_porte.csv", encoding="utf-8-sig")
display(tab_porte.round(1))

## 5. Gráfico 1 — Expansão das escolas integrais por região

**Porcentagem sobre o quê:** escolas classificadas como integrais na região, no ano ÷ todas as escolas com EM e rendimento informado na mesma região e ano (só as dependências de `DEPS_GRAFICO1`). Assim cada região é comparada com ela mesma, sem efeito do tamanho da rede. A linha "Brasil" usa o mesmo cálculo para o país todo.

São cinco versões; escolham uma para o pôster:

- **1a** linhas, % · **1b** linhas, absoluto (estrutura do PDF)
- **1c** mapa de calor, % — mesma informação do 1a sem ser gráfico de linha; mostra ano a ano e a ordem das regiões
- **1d** barras empilhadas, absoluto — total de escolas integrais e quanto cada região contribui
- **1e** halteres 2011 → 2025, % — responde direto "onde a expansão foi maior", em pontos percentuais

In [ ]:
g1 = df[df["DEP"].isin(DEPS_GRAFICO1)].assign(INTEGRAL=lambda d: d["ENS_PRED_EM"].eq("Integral"))
tab1 = (g1.groupby(["REGIAO", "ANO"])
          .agg(escolas=("CO_ESCOLA", "size"), integrais=("INTEGRAL", "sum"))
          .reset_index())
tab1["pct_integrais"] = 100 * tab1["integrais"] / tab1["escolas"]

brasil = g1.groupby("ANO").agg(escolas=("CO_ESCOLA", "size"), integrais=("INTEGRAL", "sum"))
brasil["pct_integrais"] = 100 * brasil["integrais"] / brasil["escolas"]
pct1 = tab1.pivot(index="ANO", columns="REGIAO", values="pct_integrais").reindex(columns=REGIOES).astype(float)
abs1 = tab1.pivot(index="ANO", columns="REGIAO", values="integrais").reindex(columns=REGIOES).fillna(0).astype(float)
brasil = brasil.astype(float)
ANOS = list(pct1.index)
DEPS1_TXT = nome_deps(DEPS_GRAFICO1)

tab1.to_csv(SAIDA / "tab1_integrais_regiao_ano.csv", index=False, encoding="utf-8-sig")
brasil.to_csv(SAIDA / "tab1_integrais_brasil_ano.csv", encoding="utf-8-sig")
display(pct1.round(1).assign(Brasil=brasil["pct_integrais"].round(1)))
display(abs1.astype(int).assign(Brasil=brasil["integrais"].astype(int)))

### 1a — linhas, porcentagem

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))
sombra_pandemia(ax)
for r in REGIOES:
    ax.plot(pct1.index, pct1[r], marker="o", color=COR_REGIAO[r], label=r)
if MOSTRAR_BRASIL:
    ax.plot(brasil.index, brasil["pct_integrais"], color=TINTA_2, lw=1.4, ls=(0, (4, 2)), label=T["brasil"])
eixo_anos(ax, ANOS)
ax.set_ylim(0, None)
ax.yaxis.set_major_formatter(PCT)
ax.set_xlabel(T["ano"]); ax.set_ylabel(T["g1_y_pct"])
ax.set_title(T["g1_titulo_pct"].format(escolas=DEPS1_TXT), loc="left", pad=30)
ultimo = ANOS[-1]
rotulos_finais(ax, ultimo, [pct1.loc[ultimo, r] for r in REGIOES],
               [f"{r} {pct1.loc[ultimo, r]:.1f}%" for r in REGIOES])
ax.legend(ncol=6, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9, borderaxespad=0.2)
nota(fig, T["g1_nota_pct"].format(escolas=DEPS1_TXT) + (" " + "Shading: 2020–2021." if IDIOMA == "en" else " Sombra: 2020–2021."))
salvar(fig, "g1a_integrais_regiao_pct_linhas")
plt.show()

### 1b — linhas, número absoluto

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))
sombra_pandemia(ax)
for r in REGIOES:
    ax.plot(abs1.index, abs1[r], marker="o", color=COR_REGIAO[r], label=r)
eixo_anos(ax, ANOS)
ax.set_ylim(0, None)
ax.yaxis.set_major_formatter(MIL)
ax.set_xlabel(T["ano"]); ax.set_ylabel(T["g1_y_abs"])
ax.set_title(T["g1_titulo_abs"].format(escolas=DEPS1_TXT), loc="left", pad=30)
rotulos_finais(ax, ultimo, [abs1.loc[ultimo, r] for r in REGIOES],
               [f"{r} {abs1.loc[ultimo, r]:,.0f}" for r in REGIOES])
ax.legend(ncol=5, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9, borderaxespad=0.2)
nota(fig, T["g1_nota_abs"].format(escolas=DEPS1_TXT))
salvar(fig, "g1b_integrais_regiao_abs_linhas")
plt.show()

### 1c — mapa de calor, porcentagem

Regiões ordenadas pelo valor do último ano. Cor mais escura = maior proporção de escolas integrais.

In [ ]:
ordem = pct1.iloc[-1].sort_values(ascending=False).index.tolist()
linhas = ordem + ([T["brasil"]] if MOSTRAR_BRASIL else [])
matriz = pct1[ordem].T
if MOSTRAR_BRASIL:
    matriz.loc[T["brasil"]] = brasil["pct_integrais"]
matriz = matriz.astype(float)

cmap = LinearSegmentedColormap.from_list("azul", RAMPA_AZUL)
vmax = np.nanmax(matriz.to_numpy())
fig, ax = plt.subplots(figsize=(11, 0.55 * len(linhas) + 1.6))
im = ax.imshow(matriz.to_numpy(), aspect="auto", cmap=cmap, vmin=0, vmax=vmax)
ax.set_xticks(range(len(ANOS)), ANOS, fontsize=9)
ax.set_yticks(range(len(linhas)), linhas)
ax.tick_params(length=0)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
if MOSTRAR_BRASIL:
    ax.axhline(len(ordem) - 0.5, color="white", lw=3)
for i in range(len(linhas)):
    for j in range(len(ANOS)):
        v = matriz.iat[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.0f}" if v >= 10 else f"{v:.1f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.55 * vmax else TINTA)
# marca a pandemia com um traço abaixo dos anos
j0, j1 = ANOS.index(ANOS_PANDEMIA[0]), ANOS.index(ANOS_PANDEMIA[1])
y_traco = len(linhas) - 0.5 + 0.62
ax.plot([j0 - 0.45, j1 + 0.45], [y_traco, y_traco], color=APAGADO, lw=1, clip_on=False, scalex=False, scaley=False)
ax.text((j0 + j1) / 2, y_traco + 0.06, T["pandemia"], ha="center", va="top", fontsize=8, color=APAGADO)
ax.set_ylim(len(linhas) - 0.5, -0.5)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.outline.set_visible(False)
cb.ax.tick_params(labelsize=8, length=0)
cb.ax.yaxis.set_major_formatter(PCT)
ax.set_title(T["g1_heat_titulo"].format(escolas=DEPS1_TXT), loc="left", pad=10)
nota(fig, T["g1_nota_pct"].format(escolas=DEPS1_TXT), y=-0.03)
salvar(fig, "g1c_integrais_regiao_pct_calor")
plt.show()

### 1d — barras empilhadas, número absoluto

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))
sombra_pandemia(ax)
base = np.zeros(len(ANOS))
for r in REGIOES:
    ax.bar(ANOS, abs1[r].to_numpy(), bottom=base, width=0.72, color=COR_REGIAO[r], label=r,
           edgecolor="white", linewidth=1)
    base += abs1[r].to_numpy()
for a in (ANOS[0], ANOS_PANDEMIA[0] - 1, ANOS[-1]):      # total só no início, antes da pandemia e no fim
    if a in ANOS:
        ax.text(a, base[ANOS.index(a)], f"{base[ANOS.index(a)]:,.0f}", ha="center", va="bottom",
                fontsize=9, color=TINTA_2)
eixo_anos(ax, ANOS)
ax.yaxis.set_major_formatter(MIL)
ax.set_xlabel(T["ano"]); ax.set_ylabel(T["g1_y_abs"])
ax.set_title(T["g1_titulo_abs"].format(escolas=DEPS1_TXT), loc="left")
h, l = ax.get_legend_handles_labels()
ax.legend(h[::-1], l[::-1], loc="upper left", fontsize=9, title=T["regiao"], title_fontsize=9)
nota(fig, T["g1_nota_abs"].format(escolas=DEPS1_TXT))
salvar(fig, "g1d_integrais_regiao_abs_barras")
plt.show()

### 1e — halteres: primeiro vs último ano, porcentagem

In [ ]:
a0, a1 = ANOS_HALTERES
halt = pd.DataFrame({"ini": pct1.loc[a0], "fim": pct1.loc[a1]})
if MOSTRAR_BRASIL:
    halt.loc[T["brasil"]] = [brasil.loc[a0, "pct_integrais"], brasil.loc[a1, "pct_integrais"]]
regs = halt.drop(index=T["brasil"], errors="ignore").sort_values("fim").index.tolist()
ordem_h = ([T["brasil"]] if MOSTRAR_BRASIL else []) + regs          # Brasil embaixo, maior no topo
halt = halt.loc[ordem_h]

fig, ax = plt.subplots(figsize=(9, 0.6 * len(ordem_h) + 1.6))
y = np.arange(len(ordem_h))
for yi, (nome, lin) in zip(y, halt.iterrows()):
    cor = COR_REGIAO.get(nome, TINTA_2)
    ax.plot([lin["ini"], lin["fim"]], [yi, yi], color=cor, lw=2.5, alpha=0.35, solid_capstyle="round")
    ax.scatter(lin["ini"], yi, s=55, color="white", edgecolor=cor, linewidth=2, zorder=3)
    ax.scatter(lin["fim"], yi, s=70, color=cor, edgecolor="white", linewidth=1.5, zorder=3)
    ax.text(lin["fim"] + 0.6, yi, f"{lin['fim']:.1f}%  (+{lin['fim'] - lin['ini']:.1f} {T['pp']})",
            va="center", fontsize=9, color=TINTA_2)
if MOSTRAR_BRASIL:
    ax.axhline(0.5, color=GRADE, lw=1)
ax.set_yticks(y, ordem_h)
ax.set_xlim(0, halt["fim"].max() * 1.35)
ax.xaxis.set_major_formatter(PCT)
ax.grid(axis="x", color=GRADE); ax.grid(axis="y", visible=False)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
ax.set_xlabel(T["g1_halter_x"])
ax.set_title(T["g1_halter_titulo"].format(escolas=DEPS1_TXT, a0=a0, a1=a1), loc="left")
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", mfc="white", mec=TINTA_2, mew=2, ms=7, label=str(a0)),
                   plt.Line2D([], [], marker="o", ls="", color=TINTA_2, ms=8, label=str(a1))],
          loc="lower right", fontsize=9)
nota(fig, T["g1_nota_pct"].format(escolas=DEPS1_TXT), y=-0.02)
salvar(fig, "g1e_integrais_regiao_pct_halteres")
plt.show()

## 6. Gráfico 2 — Reprovação e abandono por modalidade, ano a ano

**Porcentagem sobre o quê:** para cada modalidade e ano, soma dos reprovados (ou que abandonaram) de todas as escolas ÷ soma dos alunos com situação final (aprovados + reprovados + abandonos) dessas escolas. É a fórmula da taxa do INEP aplicada ao conjunto, e equivale à média das taxas das escolas ponderada pelo número de alunos.

**Absoluto:** soma das quantidades estimadas. Como a modalidade Regular tem muito mais alunos, ela domina essa versão; a comparação entre modalidades é a da versão em %.

> A categoria **Misto** (48–52% de matrículas integrais) é muito pequena — confiram o número de escolas-ano na tabela da seção 3 antes de interpretar essa linha.

In [ ]:
g2 = df[df["DEP"].isin(DEPS_GRAFICO2)]
tab2 = (g2.groupby(["ENS_PRED_EM", "ANO"])
          .agg(escolas=("CO_ESCOLA", "size"), alunos=("ALUNOS", "sum"),
               reprovados=("QT_REP_EM_TOTAL", "sum"), abandonos=("QT_ABN_EM_TOTAL", "sum"))
          .reset_index())
tab2["TX_REP"] = 100 * tab2["reprovados"] / tab2["alunos"]
tab2["TX_ABN"] = 100 * tab2["abandonos"] / tab2["alunos"]
tab2.to_csv(SAIDA / "tab2_taxas_modalidade_ano.csv", index=False, encoding="utf-8-sig")
DEPS2_TXT = nome_deps(DEPS_GRAFICO2)

display(tab2.pivot(index="ANO", columns="ENS_PRED_EM", values=["TX_REP", "TX_ABN"])
            .reindex(columns=MODALIDADES, level=1).round(2))


def grafico2(valor, nome_arquivo, titulo, rotulo_y, nota_txt, formato_fim, escala=1.0):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
    finais = {}
    for ax, (col_tx, col_abs) in zip(axes, [("TX_REP", "reprovados"), ("TX_ABN", "abandonos")]):
        col = col_tx if valor == "pct" else col_abs
        sombra_pandemia(ax)
        finais[ax] = ([], [])
        for m in MODALIDADES:
            s = tab2[tab2["ENS_PRED_EM"] == m].set_index("ANO")[col] / escala
            if s.empty:
                continue
            ax.plot(s.index, s, marker="o", color=COR_MOD[m], label=T["mod"][m])
            finais[ax][0].append(s.iloc[-1])
            finais[ax][1].append(formato_fim(T["mod"][m], s.iloc[-1]))
        eixo_anos(ax, ANOS)
        ax.set_ylim(0, None)
        ax.set_title(T["g2_sub"].format(taxa=T["taxa"][col_tx]), loc="left", fontsize=11)
        ax.set_xlabel(T["ano"])
    axes[0].set_ylabel(rotulo_y)
    for ax, (vals, txts) in finais.items():          # rótulos só depois de a escala comum estar fixa
        rotulos_finais(ax, ANOS[-1], vals, txts, folga=0.06)
    if valor == "pct":
        axes[0].yaxis.set_major_formatter(PCT)
    else:
        axes[0].yaxis.set_major_formatter(MIL)
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", bbox_to_anchor=(0.99, 1.0), ncol=3, fontsize=9)
    fig.suptitle(titulo, x=0.01, ha="left", fontsize=12, fontweight="bold", color=TINTA)
    fig.tight_layout(rect=(0, 0, 1, 0.97), w_pad=4)
    nota(fig, nota_txt, y=0.0)
    salvar(fig, nome_arquivo)
    plt.show()

### 2a — porcentagem

In [ ]:
grafico2("pct", "g2a_taxas_modalidade_pct", T["g2_titulo_pct"], T["g2_y_pct"],
         T["g2_nota_pct"].format(escolas=DEPS2_TXT), lambda m, v: f"{m} {v:.1f}%")

### 2b — número absoluto de alunos

In [ ]:
grafico2("abs", "g2b_taxas_modalidade_abs", T["g2_titulo_abs"], T["g2_y_abs"],
         T["g2_nota_abs"], lambda m, v: f"{m} {v:,.1f}k", escala=1000)

## 7. Gráfico 3 — Boxplots pré (2016–2019) e pós-pandemia (2022–2025)

Linhas: reprovação e abandono. Colunas: dependência administrativa. Em cada painel, as três modalidades, com a caixa clara (pré) ao lado da escura (pós).

Ajustes em relação ao PDF:
- reprovação e abandono juntos, cada linha com a própria escala (o eixo y vai até o maior bigode da linha, então nada fica cortado);
- pesos por número de alunos (ver seção 4) — configurável em `PESO_BOXPLOT`;
- caixas com poucas unidades ficam hachuradas e mostram o n.

Os quartis ponderados são calculados à mão (o `boxplot` do matplotlib não aceita pesos); o desenho usa `ax.bxp` com essas estatísticas.

In [ ]:
def periodo_do_ano(ano):
    for k, (a, b) in PERIODOS.items():
        if a <= ano <= b:
            return k
    return None

base3 = df.assign(PERIODO=df["ANO"].map(periodo_do_ano))
base3 = base3[base3["PERIODO"].notna() & (base3["ALUNOS"] > 0) & (base3["MAT_EM"] >= MIN_MATRICULAS_BOXPLOT)]

if UNIDADE_BOX == "escola-periodo":
    # taxa acumulada da escola no período (se a escola mudou de modalidade no meio, vira duas unidades)
    base3 = (base3.groupby(["CO_ESCOLA", "PERIODO", "DEP", "ENS_PRED_EM"])
                  [["ALUNOS", "QT_REP_EM_TOTAL", "QT_ABN_EM_TOTAL"]].sum().reset_index())
    base3["TX_REP"] = 100 * base3["QT_REP_EM_TOTAL"] / base3["ALUNOS"]
    base3["TX_ABN"] = 100 * base3["QT_ABN_EM_TOTAL"] / base3["ALUNOS"]


def quantis(v, w, qs):
    """Quantis ponderados (interpolação linear). Sem peso, igual ao np.percentile."""
    if w is None:
        return np.percentile(v, np.array(qs) * 100)
    o = np.argsort(v)
    v, w = v[o], w[o]
    acum = np.cumsum(w)
    p = (acum - 0.5 * w) / acum[-1]
    return np.interp(qs, p, v)


def estat_caixa(v, w=None):
    v = np.asarray(v, dtype=float)
    w = None if w is None else np.asarray(w, dtype=float)
    q1, med, q3 = quantis(v, w, [0.25, 0.5, 0.75])
    iqr = q3 - q1
    return {"q1": q1, "med": med, "q3": q3,
            "whislo": v[v >= q1 - 1.5 * iqr].min(), "whishi": v[v <= q3 + 1.5 * iqr].max(),
            "fliers": [], "n": len(v), "media_pond": np.average(v, weights=w)}


estats = []
for taxa in ("TX_REP", "TX_ABN"):
    for dep in DEPS_TODAS:
        for mod in MODALIDADES:
            for per in PERIODOS:
                s = base3[(base3["DEP"] == dep) & (base3["ENS_PRED_EM"] == mod) & (base3["PERIODO"] == per)]
                if s.empty:
                    continue
                w = s["ALUNOS"].to_numpy() if PESO_BOXPLOT == "alunos" else None
                estats.append({"taxa": taxa, "dep": dep, "mod": mod, "periodo": per,
                               **estat_caixa(s[taxa].to_numpy(), w)})
tab3 = pd.DataFrame(estats)
tab3.drop(columns="fliers").to_csv(SAIDA / "tab3_boxplots.csv", index=False, encoding="utf-8-sig")
display(tab3.drop(columns="fliers").set_index(["taxa", "dep", "mod", "periodo"]).round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharey="row")
desloc = {"pre": -0.2, "pos": 0.2}
x_mod = np.arange(len(MODALIDADES))
unid_txt = T["unidade"][UNIDADE_BOX]

for i, taxa in enumerate(("TX_REP", "TX_ABN")):
    linha = tab3[tab3["taxa"] == taxa]
    topo = linha["whishi"].max() if not linha.empty else 1
    for j, dep in enumerate(DEPS_TODAS):
        ax = axes[i, j]
        for per in PERIODOS:
            sel = linha[(linha["dep"] == dep) & (linha["periodo"] == per)]
            if sel.empty:
                continue
            pos = [x_mod[MODALIDADES.index(m)] + desloc[per] for m in sel["mod"]]
            caixas = ax.bxp(sel.to_dict("records"), positions=pos, widths=0.34, patch_artist=True,
                            showfliers=False, manage_ticks=False,
                            medianprops=dict(color=TINTA if per == "pre" else "white", lw=1.6),
                            whiskerprops=dict(color=APAGADO, lw=1), capprops=dict(color=APAGADO, lw=1),
                            boxprops=dict(lw=0))
            for patch, mediana, n, p in zip(caixas["boxes"], caixas["medians"], sel["n"], pos):
                patch.set_facecolor(COR_PERIODO[per])
                if n < MIN_N_CELULA:
                    patch.set_alpha(0.45); patch.set_hatch("////"); patch.set_edgecolor(TINTA_2)
                    mediana.set_color(TINTA)
                    ax.text(p, -0.04 * topo, f"n={n}", ha="center", va="top", fontsize=7, color=APAGADO)
        ax.set_xticks(x_mod, [T["mod"][m] for m in MODALIDADES])
        ax.tick_params(axis="x", length=0, pad=12)
        ax.set_xlim(-0.6, len(MODALIDADES) - 0.4)
        ax.set_ylim(-0.12 * topo, topo * 1.06)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, steps=[1, 2, 5, 10]))
        ax.yaxis.set_major_formatter(PCT)
        ax.spines["bottom"].set_visible(False)
        ax.axhline(0, color=EIXO, lw=0.8, zorder=1)
        if i == 0:
            ax.set_title(T["dep"][dep], fontsize=11)
        if j == 0:
            ax.set_ylabel(T["g3_y"].format(taxa=T["taxa"][taxa]))

legenda = [Patch(facecolor=COR_PERIODO[p], label=f"{T['periodo'][p]} ({a}–{b})") for p, (a, b) in PERIODOS.items()]
legenda.append(Patch(facecolor="white", edgecolor=TINTA_2, hatch="////", label=T["n_baixo"].format(n=MIN_N_CELULA)))
fig.legend(handles=legenda, loc="upper left", ncol=3, bbox_to_anchor=(0.005, 0.965), fontsize=9)
fig.suptitle(T["g3_titulo"], x=0.01, ha="left", y=0.995, fontsize=12, fontweight="bold", color=TINTA)
fig.tight_layout(rect=(0, 0, 1, 0.93), h_pad=2.5)
texto_peso = T["g3_nota_peso"] if PESO_BOXPLOT == "alunos" else T["g3_nota_sem_peso"].format(unidade=unid_txt)
nota(fig, texto_peso + T["g3_nota_fim"].format(n=MIN_N_CELULA, unidade=unid_txt), y=0.0)
salvar(fig, "g3_boxplots_rep_abn_pre_pos")
plt.show()

### Sensibilidade da escolha do peso

Medianas de cada caixa em três versões. Se as diferenças entre modalidades e entre períodos apontam na mesma direção nas três colunas, a conclusão do gráfico não depende da escolha do peso.

In [ ]:
b = df.assign(PERIODO=df["ANO"].map(periodo_do_ano))
b = b[b["PERIODO"].notna() & (b["ALUNOS"] > 0)]
linhas_sens = []
for (dep, mod, per), s in b.groupby(["DEP", "ENS_PRED_EM", "PERIODO"]):
    grande = s[s["MAT_EM"] >= 51]
    for taxa in ("TX_REP", "TX_ABN"):
        linhas_sens.append({
            "taxa": taxa, "dep": dep, "mod": mod, "periodo": per,
            "sem peso (todas)": np.median(s[taxa]),
            "sem peso (≥ 51 matr.)": np.median(grande[taxa]) if len(grande) else np.nan,
            "peso = alunos": quantis(s[taxa].to_numpy(), s["ALUNOS"].to_numpy(), [0.5])[0],
            "n": len(s), "n (≥ 51)": len(grande),
        })
sens = (pd.DataFrame(linhas_sens)
          .assign(mod=lambda d: pd.Categorical(d["mod"], MODALIDADES),
                  dep=lambda d: pd.Categorical(d["dep"], DEPS_TODAS))
          .sort_values(["taxa", "dep", "mod", "periodo"], ascending=[False, True, True, False])
          .set_index(["taxa", "dep", "mod", "periodo"]))
sens.to_csv(SAIDA / "tab3_sensibilidade_peso.csv", encoding="utf-8-sig")
display(sens.round(2))

## 8. Arquivos gerados

In [ ]:
for p in sorted(SAIDA.iterdir()):
    print(p.name)